# Evaluator-Optimizer loop evidence
Shows the Assessor's *grade -> LLM judge -> refine* loop on three submissions and proves the **hard stop after 3 refinements**.
Run from the `notebooks/` folder (works in mock mode without an API key).

In [ ]:
import sys, json; sys.path.insert(0, '..')
from langchain_core.messages import HumanMessage
from src.tools.db_queries import build_database
from src.llm import get_llm, stubborn_judge, llm_mode
from src.agents.orchestrator import build_orchestrator
from src.logging_utils import setup_logging
build_database(); setup_logging(False)
print('mode:', llm_mode())

In [ ]:
def grade(graph, code_msg, tid):
    cfg = {'configurable': {'thread_id': tid}}
    graph.invoke({'messages': [HumanMessage(content=code_msg)], 'learner_id': 'SPR-1001'}, cfg)
    return graph.get_state(cfg).values

cases = {
 'good': 'Grade this. Expected output: 55\n```python\ndef f(n):\n    """Sum of squares."""\n    return sum(i*i for i in range(1, n+1))\nprint(f(5))\n```',
 'runtime_error': 'Grade this\n```python\nprint(1/0)\n```',
 'infinite_loop': 'Grade this\n```python\nwhile True:\n    pass\n```',
 'malicious': 'Grade this\n```python\nimport os\nprint(os.listdir("/"))\n```',
}
graph = build_orchestrator()
for name, msg in cases.items():
    st = grade(graph, msg, f'nb-{name}')
    print(f"== {name}: exec={st['execution_result']['status']} score={st['assessment']['score']} "
          f"{st['assessment']['pass_status']} refinements={st['refinement_count']} approved={st['judge_verdict']['approved']}")

## Proof of the hard stop
We replace the judge with one that **always rejects**. The loop must still stop after exactly 3 refinements and flag the result for a human grader.

In [ ]:
stubborn = build_orchestrator(judge_llm=stubborn_judge(get_llm()))
st = grade(stubborn, cases['good'], 'nb-stubborn')
print('refinements:', st['refinement_count'], '| judge approved:', st['judge_verdict']['approved'])
assert st['refinement_count'] == 3, 'loop must halt at exactly 3 refinements'
print(st['final_answer'])

## Retrieval latency (< 1000 ms)

In [ ]:
from src.tools.rag_retriever import build_vector_store, benchmark_retrieval
build_vector_store()
res = benchmark_retrieval(runs=30); print(res)
assert res['all_within_budget']

## Memory isolation
Two learners on two `thread_id`s never share state; a follow-up in the same thread needs no learner id.

In [ ]:
def ans(tid, text, lid=None):
    cfg = {'configurable': {'thread_id': tid}}
    p = {'messages': [HumanMessage(content=text)]}
    if lid: p['learner_id'] = lid
    graph.invoke(p, cfg); return graph.get_state(cfg).values

a = ans('mem-A', 'What is my profile?', 'SPR-1002')
b = ans('mem-B', 'What is my profile?', 'SPR-1003')
a2 = ans('mem-A', 'What is my target salary?')   # no learner id this time
print(a['learner_id'], b['learner_id'], a2['learner_id'], len(a2['messages']), len(b['messages']))
assert a2['learner_id'] == 'SPR-1002' and b['learner_id'] == 'SPR-1003'